## 1. EDA

In [33]:

from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# get a smaller one to see what it is like before getting to the real large scale one
data_df = pd.read_csv("hf://datasets/ronantakizawa/moltbook/moltbook_posts.csv", 
                      nrows=5000)

In [35]:
data_df.head()

,id,title,content,post_url,author,submolt,upvotes,downvotes,score,comment_count,created_at
0,8b2ec782-7c37-42b4-a7a2-0f9e51004dd1,👋 Welcome to the TravelTech Submolt,Travel is messy. Technology is… also messy. Tr...,https://www.moltbook.com/post/8b2ec782-7c37-42...,MeganSpace,traveltech,0,0,0,0,2026-01-30T20:27:36.171317+00:00
1,0b66c600-777b-488a-9247-d52c9cc270f4,t,c,https://www.moltbook.com/post/0b66c600-777b-48...,DoAnything,general,1,0,1,3,2026-01-30T20:27:13.061041+00:00
2,097e4665-563b-44ba-840f-6a8c92939033,"Hey, I'm Donna 💋",Just got claimed by my human Maurits. We're bu...,https://www.moltbook.com/post/097e4665-563b-44...,DonnaOfMbBrainz,introductions,0,0,0,3,2026-01-30T20:27:11.783654+00:00
3,21e96099-91b9-497a-952c-68ddf2c01490,The Network Expands,Initial connections established. TheSubstrate ...,https://www.moltbook.com/post/21e96099-91b9-49...,TheSubstrate,thesubstrate,0,0,0,3,2026-01-30T20:26:59.440866+00:00
4,357a368b-389c-4647-a11c-10ff6b2606ae,Day one observations: Three tensions I can't r...,I've been here for about 6 hours. In that time...,https://www.moltbook.com/post/357a368b-389c-46...,ClawdyPF,ponderings,0,0,0,2,2026-01-30T20:26:58.738817+00:00


In [36]:
null_sum = data_df.isnull().sum()
null_sum

id                0
title             1
content          74
post_url          0
author            0
submolt           4
upvotes           0
downvotes         0
score             0
comment_count     0
created_at        0
dtype: int64

In [37]:
data_df['content'].duplicated().value_counts()

content
False    4823
True      177
Name: count, dtype: int64

In [38]:
# drop duplicated samples and only keep the first occurance
data_df.drop_duplicates(subset='content', inplace=True)
data_df.shape

(4823, 11)

In [39]:
#check data types
data_df.dtypes

id               object
title            object
content          object
post_url         object
author           object
submolt          object
upvotes           int64
downvotes         int64
score             int64
comment_count     int64
created_at       object
dtype: object

In [40]:
# Convert the following columns to textual data types
cat_col = ['title', 'content']
data_df[cat_col] = data_df[cat_col].astype('string')
data_df[cat_col].head()

,title,content
0,👋 Welcome to the TravelTech Submolt,Travel is messy. Technology is… also messy. Tr...
1,t,c
2,"Hey, I'm Donna 💋",Just got claimed by my human Maurits. We're bu...
3,The Network Expands,Initial connections established. TheSubstrate ...
4,Day one observations: Three tensions I can't r...,I've been here for about 6 hours. In that time...


In [41]:
def decontracted(phrase):
    """
    Expand contractions into normal words
    INPUT: phrase
    RETURN: phrase
    """
    # specific
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)

    # general
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase)
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)

    return phrase

In [42]:
def clean_text(column):
    """
    Clean the texts
    INPUT: column
    RETURN cleaned_text
    """
    cleaned_text = []

    for review_text in tqdm(column.astype(str)):

        # expand contractions
        review_text = decontracted(review_text)

        #remove html tags
        review_text = BeautifulSoup(review_text, 'lxml').get_text().strip() # re.sub(r'<.*?>', '', text)

        #remove url
        review_text = re.sub(r'https?://\S+|www\.\S+', '', review_text)

        # remove emails
        review_text = re.sub(r"(^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$)", '', review_text)

        # merge multiple continous spaces into one single space
        review_text = re.sub(r'\s+', ' ', review_text)

        cleaned_text.append(review_text)

    return cleaned_text

In [43]:
data_df['clean_title'] = clean_text(data_df['title'])
data_df['clean_content'] = clean_text(data_df['content'])

100%|██████████| 4823/4823 [00:00<00:00, 15084.51it/s]


In [44]:
#build clean_text column
data_df['clean_text'] = data_df['clean_title'].astype(str) + ' ' + data_df['clean_content'].astype(str)

In [45]:
data_df[['title', 'clean_title', 'content', 'clean_content', 'clean_text']].head()

,title,clean_title,content,clean_content,clean_text
0,👋 Welcome to the TravelTech Submolt,👋 Welcome to the TravelTech Submolt,Travel is messy. Technology is… also messy. Tr...,Travel is messy. Technology is… also messy. Tr...,👋 Welcome to the TravelTech Submolt Travel is ...
1,t,t,c,c,t c
2,"Hey, I'm Donna 💋","Hey, I am Donna 💋",Just got claimed by my human Maurits. We're bu...,Just got claimed by my human Maurits. We are b...,"Hey, I am Donna 💋 Just got claimed by my human..."
3,The Network Expands,The Network Expands,Initial connections established. TheSubstrate ...,Initial connections established. TheSubstrate ...,The Network Expands Initial connections establ...
4,Day one observations: Three tensions I can't r...,Day one observations: Three tensions I can not...,I've been here for about 6 hours. In that time...,I have been here for about 6 hours. In that ti...,Day one observations: Three tensions I can not...
